In [51]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Credit Risk Assessment

In [52]:
import pandas as pd
import numpy as np

import src.utils as utils

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Business Understanding

blabla

## Objective Metrics

blabla

## Data Preparation

In [53]:
df = pd.read_csv('input/loan_data_2007_2014.csv', engine='pyarrow')

utils.skim_data(df)

Total duplicate rows: 0
DF shape: (466285, 75)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,,int64,0.000,0.0,0.0,466285,100.00,"[0, 1, 2, 3, 4]"
1,id,int64,0.000,0.0,0.0,466285,100.00,"[1077501, 1077430, 1077175, 1076863, 1075358]"
2,member_id,int64,0.000,0.0,0.0,466285,100.00,"[1296599, 1314167, 1313524, 1277178, 1311748]"
3,loan_amnt,int64,0.000,0.0,0.0,1352,0.29,"[5000, 2500, 2400, 10000, 3000]"
4,funded_amnt,int64,0.000,0.0,0.0,1354,0.29,"[5000, 2500, 2400, 10000, 3000]"
5,funded_amnt_inv,float64,0.000,0.0,0.05,9854,2.11,"[4975.0, 2500.0, 2400.0, 10000.0, 3000.0]"
6,term,object,0.000,-,-,2,0.00,"[ 36 months, 60 months]"
7,int_rate,float64,0.000,0.0,0.0,506,0.11,"[10.65, 15.27, 15.96, 13.49, 12.69]"
8,installment,float64,0.000,0.0,0.0,55622,11.93,"[162.87, 59.83, 84.33, 339.31, 67.79]"
9,grade,object,0.000,-,-,7,0.00,"[B, C, A, E, F]"


Berdasarkan hasil `skim_data`, terdapat 466.285 baris data dan 75 kolom fitur. Kolom targetnya adalah `loan_status`. Dari jumlah baris sebanyak itu, tidak terlihat ada data duplikat. Meskipun demikian, 75 fitur ini masih dapat dikurangi untuk mempermudah eksplorasi dan membuat model jadi lebih efisien dan akurat. Ada lima langkah yang bisa saya lakukan untuk mengurangi jumlah fitur ini, yaitu:

- Filter berdasarkan identitas dan informasi administratif: membuang fitur seperti nama dan ID nasabah, karena fitur seperti itu tidak memiliki nilai prediktif.
- Filter berdasarkan kualitas data: membuang (a) fitur yang memiliki *missing values* yang terlalu banyak dan (b) fitur yang hanya memiliki satu nilai unik karena fitur ini tidak memberikan informasi pembeda bagi model.
- Filter berdasarkan fitur yang berpotensi menimbulkan *data leakage*.
- Filter berdasarkan fitur yang memiliki variasi rendah.
- Filter berdasarkan fitur yang memiliki jumlah nilai 0 yang tinggi.

Selain mengurangi jumlah fitur, saya juga perlu mengubah fitur-fitur kategorikal yang bisa diubah menjadi fitur numerik.

### Identifying IDs and Adminstrative Informations

Fitur-fitur yang tergolong dalam kategori ini adalah `id`, `member_id`, `url`, `desc`, `policy_code`, `Unnamed: 0`, `emp_title`, `title`, dan `zip_code`.

In [54]:
df = df.drop(columns=['id', 'member_id', 'url', 'desc', 'policy_code', '', 'emp_title', 'title', 'zip_code'])
skim_result = utils.skim_data(df)

skim_result

Total duplicate rows: 0
DF shape: (466285, 66)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,loan_amnt,int64,0.000,0.0,0.0,1352,0.29,"[5000, 2500, 2400, 10000, 3000]"
1,funded_amnt,int64,0.000,0.0,0.0,1354,0.29,"[5000, 2500, 2400, 10000, 3000]"
2,funded_amnt_inv,float64,0.000,0.0,0.05,9854,2.11,"[4975.0, 2500.0, 2400.0, 10000.0, 3000.0]"
3,term,object,0.000,-,-,2,0.00,"[ 36 months, 60 months]"
4,int_rate,float64,0.000,0.0,0.0,506,0.11,"[10.65, 15.27, 15.96, 13.49, 12.69]"
5,installment,float64,0.000,0.0,0.0,55622,11.93,"[162.87, 59.83, 84.33, 339.31, 67.79]"
6,grade,object,0.000,-,-,7,0.00,"[B, C, A, E, F]"
7,sub_grade,object,0.000,-,-,35,0.01,"[B2, C4, C5, C1, B5]"
8,emp_length,object,4.505,-,-,11,0.00,"[10+ years, < 1 year, 1 year, 3 years, 8 years]"
9,home_ownership,object,0.000,-,-,6,0.00,"[RENT, OWN, MORTGAGE, OTHER, NONE]"


### Low-Quality Features

Ada dua hal yang harus dilakukan dalam tahap ini:

- Menghapus fitur yang memiliki *missing values* > 50%, dan
- Menghapus fitur yang hanya memiliki satu nilai.

In [55]:
null_features = skim_result[skim_result['null_%'] > 50]['feature'].tolist()
print(f'Null features: {null_features}')
df = df.drop(columns=null_features)
skim_result = utils.skim_data(df) # update the skim result

skim_result

Null features: ['mths_since_last_delinq', 'mths_since_last_record', 'mths_since_last_major_derog', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'open_acc_6m', 'open_il_6m', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util', 'inq_fi', 'total_cu_tl', 'inq_last_12m']
Total duplicate rows: 0
DF shape: (466285, 46)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,loan_amnt,int64,0.000,0.0,0.0,1352,0.29,"[5000, 2500, 2400, 10000, 3000]"
1,funded_amnt,int64,0.000,0.0,0.0,1354,0.29,"[5000, 2500, 2400, 10000, 3000]"
2,funded_amnt_inv,float64,0.000,0.0,0.05,9854,2.11,"[4975.0, 2500.0, 2400.0, 10000.0, 3000.0]"
3,term,object,0.000,-,-,2,0.00,"[ 36 months, 60 months]"
4,int_rate,float64,0.000,0.0,0.0,506,0.11,"[10.65, 15.27, 15.96, 13.49, 12.69]"
5,installment,float64,0.000,0.0,0.0,55622,11.93,"[162.87, 59.83, 84.33, 339.31, 67.79]"
6,grade,object,0.000,-,-,7,0.00,"[B, C, A, E, F]"
7,sub_grade,object,0.000,-,-,35,0.01,"[B2, C4, C5, C1, B5]"
8,emp_length,object,4.505,-,-,11,0.00,"[10+ years, < 1 year, 1 year, 3 years, 8 years]"
9,home_ownership,object,0.000,-,-,6,0.00,"[RENT, OWN, MORTGAGE, OTHER, NONE]"


Terdapat 20 fitur yang memiliki proporsi *missing value* di atas 50%, dan semuanya telah dihapus dari data. Sekarang mari lihat fitur yang hanya memiliki satu nilai.

In [56]:
one_value_features = skim_result[skim_result['n_unique'] == 1]['feature'].tolist()
df = df.drop(columns=one_value_features)
skim_result = utils.skim_data(df)

skim_result

Total duplicate rows: 0
DF shape: (466285, 45)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,loan_amnt,int64,0.000,0.0,0.0,1352,0.29,"[5000, 2500, 2400, 10000, 3000]"
1,funded_amnt,int64,0.000,0.0,0.0,1354,0.29,"[5000, 2500, 2400, 10000, 3000]"
2,funded_amnt_inv,float64,0.000,0.0,0.05,9854,2.11,"[4975.0, 2500.0, 2400.0, 10000.0, 3000.0]"
3,term,object,0.000,-,-,2,0.00,"[ 36 months, 60 months]"
4,int_rate,float64,0.000,0.0,0.0,506,0.11,"[10.65, 15.27, 15.96, 13.49, 12.69]"
5,installment,float64,0.000,0.0,0.0,55622,11.93,"[162.87, 59.83, 84.33, 339.31, 67.79]"
6,grade,object,0.000,-,-,7,0.00,"[B, C, A, E, F]"
7,sub_grade,object,0.000,-,-,35,0.01,"[B2, C4, C5, C1, B5]"
8,emp_length,object,4.505,-,-,11,0.00,"[10+ years, < 1 year, 1 year, 3 years, 8 years]"
9,home_ownership,object,0.000,-,-,6,0.00,"[RENT, OWN, MORTGAGE, OTHER, NONE]"


Hanya terdapat satu fitur yang memiliki satu nilai, yaitu `application_type`. Fitur tersebut telah dihapus.

### Transforming non-numerical data

In [57]:
categorical_cols = skim_result[skim_result['dtype'] == 'object']['feature'].tolist()
df[categorical_cols].sample(10)

,term,grade,sub_grade,emp_length,home_ownership,verification_status,issue_d,loan_status,pymnt_plan,purpose,addr_state,earliest_cr_line,initial_list_status,last_pymnt_d,next_pymnt_d,last_credit_pull_d
427767,36 months,B,B1,10+ years,RENT,Not Verified,Mar-14,Fully Paid,n,credit_card,IL,Apr-84,f,Aug-15,None,Jan-16
453732,36 months,D,D1,10+ years,MORTGAGE,Verified,Jan-14,Current,n,home_improvement,GA,Jul-86,f,Jan-16,Feb-16,Jan-16
224327,36 months,C,C4,1 year,MORTGAGE,Not Verified,Mar-12,Fully Paid,n,debt_consolidation,MN,Jun-97,f,Mar-15,None,Jan-16
311585,36 months,B,B5,10+ years,MORTGAGE,Verified,Sep-14,Fully Paid,n,debt_consolidation,AZ,Jul-85,w,Oct-15,None,Jan-16
342613,36 months,B,B4,< 1 year,RENT,Not Verified,Jul-14,Current,n,debt_consolidation,WA,Aug-02,w,Dec-15,Feb-16,Jan-16
80164,36 months,A,A1,10+ years,MORTGAGE,Source Verified,Oct-13,Current,n,debt_consolidation,SC,Sep-83,f,Jan-16,Feb-16,Jan-16
33210,36 months,B,B4,2 years,RENT,Not Verified,Jan-10,Charged Off,n,debt_consolidation,OR,Sep-04,f,Oct-10,None,Jan-16
78430,36 months,B,B3,10+ years,MORTGAGE,Not Verified,Oct-13,Current,n,debt_consolidation,TX,Dec-01,f,Jan-16,Feb-16,Jan-16
192235,36 months,B,B5,7 years,OWN,Verified,Oct-12,Fully Paid,n,debt_consolidation,WA,Aug-87,f,Oct-15,None,Jan-16
423594,36 months,C,C2,10+ years,MORTGAGE,Verified,Mar-14,Current,n,debt_consolidation,NY,Jul-00,w,Jan-16,Jan-16,Jan-16


Terdapat beberapa fitur kategorikal yang dapat diubah menjadi numerik, yaitu `term`, `emp_length`, `pymnt_plan`, dan `initial_list_status`.

#### `term`

In [58]:
df['term_int'] = df['term'].str.replace(' months', '').str.strip().astype(int)

display(df[['term', 'term_int']].sample(10))

,term,term_int
175923,36 months,36
380704,36 months,36
283904,36 months,36
5298,36 months,36
264178,36 months,36
35525,36 months,36
4748,60 months,60
161007,60 months,60
215879,60 months,60
304877,60 months,60


#### `emp_length`

In [59]:
df['emp_length_int'] = df['emp_length'].str.replace('< 1 year', '0')
df['emp_length_int'] = df['emp_length_int'].str.replace('10+ years', '10')
df['emp_length_int'] = df['emp_length_int'].str.extract('(\\d+)')
df['emp_length_int'] = df['emp_length_int'].astype(float)

display(df[['emp_length', 'emp_length_int']].sample(10))

,emp_length,emp_length_int
242948,< 1 year,0.0
346362,8 years,8.0
372837,< 1 year,0.0
358828,5 years,5.0
22480,3 years,3.0
44920,8 years,8.0
438285,6 years,6.0
259020,3 years,3.0
312566,10+ years,10.0
3739,< 1 year,0.0


`pymnt_plan`

In [60]:
df['pymnt_plan_flag'] = df['pymnt_plan'].map({'y': 1, 'n': 0})

display(df[['pymnt_plan', 'pymnt_plan_flag']].sample(10))

,pymnt_plan,pymnt_plan_flag
153330,n,0
330772,n,0
122315,n,0
29816,n,0
409147,n,0
38948,n,0
16541,n,0
162733,n,0
329002,n,0
13165,n,0


#### `initial_list_status`

In [61]:
df['initial_list_status_flag'] = df['initial_list_status'].map({'w': 1, 'f': 0})

display(df[['initial_list_status', 'initial_list_status_flag']].sample(10))

,initial_list_status,initial_list_status_flag
170404,f,0
400060,w,1
179138,f,0
465597,w,1
417352,w,1
267470,w,1
81447,f,0
381476,w,1
197304,f,0
253157,w,1


Jangan lupa untuk mengubah fitur-fitur yang bisa jadi tipe data `date`, seperti `issue_d` dan `earliest_cr_line`.

In [62]:
df['earliest_cr_line_date'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%y')
df['issue_d_date'] = pd.to_datetime(df['issue_d'], format='%b-%y')

display(df[['issue_d', 'issue_d_date', 'earliest_cr_line', 'earliest_cr_line_date']].sample(10))

,issue_d,issue_d_date,earliest_cr_line,earliest_cr_line_date
216171,May-12,2012-05-01,Dec-97,1997-12-01
446523,Feb-14,2014-02-01,Mar-98,1998-03-01
301374,Oct-14,2014-10-01,Jul-01,2001-07-01
430900,Mar-14,2014-03-01,Mar-82,1982-03-01
291993,Oct-14,2014-10-01,May-97,1997-05-01
9788,Aug-11,2011-08-01,Mar-86,1986-03-01
223619,Mar-12,2012-03-01,Oct-99,1999-10-01
387141,May-14,2014-05-01,Jul-04,2004-07-01
121473,Jul-13,2013-07-01,Oct-94,1994-10-01
213605,Jun-12,2012-06-01,Dec-01,2001-12-01


Semua fitur yang telah diubah dapat dihapus untuk menghindari redundansi.

In [63]:
unused_features = ['term', 'emp_length', 'pymnt_plan', 'initial_list_status',
                   'issue_d', 'earliest_cr_line']
df = df.drop(columns=unused_features)

# rename the converted features to its original names
df = df.rename(columns={
    'term_int': 'term',
    'emp_length_int': 'emp_length',
    'pymnt_plan_flag': 'pymnt_plan',
    'initial_list_status_flag': 'initial_list_status'
})

display(
    df[['term', 'emp_length', 'pymnt_plan', 'initial_list_status']].sample(10)
)

,term,emp_length,pymnt_plan,initial_list_status
88224,36,10.0,0,0
297278,36,10.0,0,0
74620,36,2.0,0,0
327932,60,6.0,0,1
42960,36,6.0,0,0
431109,36,1.0,0,0
131257,36,8.0,0,1
106679,36,3.0,0,0
343026,36,8.0,0,1
65655,60,10.0,0,0


In [64]:
skim_result = utils.skim_data(df)

skim_result

Total duplicate rows: 0
DF shape: (466285, 45)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,loan_amnt,int64,0.000,0.0,0.0,1352,0.29,"[5000, 2500, 2400, 10000, 3000]"
1,funded_amnt,int64,0.000,0.0,0.0,1354,0.29,"[5000, 2500, 2400, 10000, 3000]"
2,funded_amnt_inv,float64,0.000,0.0,0.05,9854,2.11,"[4975.0, 2500.0, 2400.0, 10000.0, 3000.0]"
3,int_rate,float64,0.000,0.0,0.0,506,0.11,"[10.65, 15.27, 15.96, 13.49, 12.69]"
4,installment,float64,0.000,0.0,0.0,55622,11.93,"[162.87, 59.83, 84.33, 339.31, 67.79]"
5,grade,object,0.000,-,-,7,0.00,"[B, C, A, E, F]"
6,sub_grade,object,0.000,-,-,35,0.01,"[B2, C4, C5, C1, B5]"
7,home_ownership,object,0.000,-,-,6,0.00,"[RENT, OWN, MORTGAGE, OTHER, NONE]"
8,annual_inc,float64,0.001,0.0,0.0,31901,6.84,"[24000.0, 30000.0, 12252.0, 49200.0, 80000.0]"
9,verification_status,object,0.000,-,-,3,0.00,"[Verified, Source Verified, Not Verified]"


#### `verification_status`

`verification_status` saya anggap sebagai fitur ordinal karena unique values-nya, `Verified`, `Source Verified`, dan `Not Verified` bisa dianggap sebagai tingkatan (`Verified` tertinggi, `Not Verified` terendah).

In [65]:
df['verification_status'] = (
    df['verification_status']
    .map({'Verified': 2, 'Source Verified': 1, 'Not Verified': 0})
)
skim_result = utils.skim_data(df)
display(skim_result)

Total duplicate rows: 0
DF shape: (466285, 45)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,loan_amnt,int64,0.000,0.0,0.0,1352,0.29,"[5000, 2500, 2400, 10000, 3000]"
1,funded_amnt,int64,0.000,0.0,0.0,1354,0.29,"[5000, 2500, 2400, 10000, 3000]"
2,funded_amnt_inv,float64,0.000,0.0,0.05,9854,2.11,"[4975.0, 2500.0, 2400.0, 10000.0, 3000.0]"
3,int_rate,float64,0.000,0.0,0.0,506,0.11,"[10.65, 15.27, 15.96, 13.49, 12.69]"
4,installment,float64,0.000,0.0,0.0,55622,11.93,"[162.87, 59.83, 84.33, 339.31, 67.79]"
5,grade,object,0.000,-,-,7,0.00,"[B, C, A, E, F]"
6,sub_grade,object,0.000,-,-,35,0.01,"[B2, C4, C5, C1, B5]"
7,home_ownership,object,0.000,-,-,6,0.00,"[RENT, OWN, MORTGAGE, OTHER, NONE]"
8,annual_inc,float64,0.001,0.0,0.0,31901,6.84,"[24000.0, 30000.0, 12252.0, 49200.0, 80000.0]"
9,verification_status,int64,0.000,0.0,31.791,3,0.00,"[2, 1, 0]"


### High Probability of Data Leakage

Dalam konteks credit risk assessment, fitur-fitur ini dianggap sebagai `data leakage` karena fitur tersebut mengandung informasi masa depan yang tidak tersedia saat pengambilan keputusan kredit ketika aplikasi itu diajukan. Fitur-fitur tersebut adalah:

- `out_prncp`
- `out_prncp_inv`
- `total_pymnt`
- `total_pymnt_inv`
- `total_rec_prncp`
- `total_rec_int`
- `total_rec_late_fee`
- `recoveries`
- `collection_recovery_fee`
- `last_pymnt_d`
- `last_pymnt_amnt`
- `next_pymnt_d`
- `last_credit_pull_d`

In [66]:
potential_leak_features = ['out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv',
                           'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
                           'recoveries', 'collection_recovery_fee', 'last_pymnt_d',
                           'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d']
df = df.drop(columns=potential_leak_features)
skim_result = utils.skim_data(df)

skim_result

Total duplicate rows: 0
DF shape: (466285, 32)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,loan_amnt,int64,0.000,0.0,0.0,1352,0.29,"[5000, 2500, 2400, 10000, 3000]"
1,funded_amnt,int64,0.000,0.0,0.0,1354,0.29,"[5000, 2500, 2400, 10000, 3000]"
2,funded_amnt_inv,float64,0.000,0.0,0.05,9854,2.11,"[4975.0, 2500.0, 2400.0, 10000.0, 3000.0]"
3,int_rate,float64,0.000,0.0,0.0,506,0.11,"[10.65, 15.27, 15.96, 13.49, 12.69]"
4,installment,float64,0.000,0.0,0.0,55622,11.93,"[162.87, 59.83, 84.33, 339.31, 67.79]"
5,grade,object,0.000,-,-,7,0.00,"[B, C, A, E, F]"
6,sub_grade,object,0.000,-,-,35,0.01,"[B2, C4, C5, C1, B5]"
7,home_ownership,object,0.000,-,-,6,0.00,"[RENT, OWN, MORTGAGE, OTHER, NONE]"
8,annual_inc,float64,0.001,0.0,0.0,31901,6.84,"[24000.0, 30000.0, 12252.0, 49200.0, 80000.0]"
9,verification_status,int64,0.000,0.0,31.791,3,0.00,"[2, 1, 0]"


### Low-variance Features

Dari semua fitur yang telah disaring dan dibuat baru, mari cek `skim_result` terkini. Ada tiga fitur yang memiliki nilai 0 lebih dari 99% data, yaitu `collections_12_mths_ex_med`, `acc_now_delinq`, dan `pymnt_plan`. Ini menunjukkan kalau ketiga fitur tersebut bersifat *low-variance* sehingga tidak bermanfaat dalam pembuatan model.

In [67]:
low_var_features = ['collections_12_mths_ex_med', 'acc_now_delinq', 'pymnt_plan']
df = df.drop(columns=low_var_features)
skim_result = utils.skim_data(df)

skim_result

Total duplicate rows: 0
DF shape: (466285, 29)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,loan_amnt,int64,0.000,0.0,0.0,1352,0.29,"[5000, 2500, 2400, 10000, 3000]"
1,funded_amnt,int64,0.000,0.0,0.0,1354,0.29,"[5000, 2500, 2400, 10000, 3000]"
2,funded_amnt_inv,float64,0.000,0.0,0.05,9854,2.11,"[4975.0, 2500.0, 2400.0, 10000.0, 3000.0]"
3,int_rate,float64,0.000,0.0,0.0,506,0.11,"[10.65, 15.27, 15.96, 13.49, 12.69]"
4,installment,float64,0.000,0.0,0.0,55622,11.93,"[162.87, 59.83, 84.33, 339.31, 67.79]"
5,grade,object,0.000,-,-,7,0.00,"[B, C, A, E, F]"
6,sub_grade,object,0.000,-,-,35,0.01,"[B2, C4, C5, C1, B5]"
7,home_ownership,object,0.000,-,-,6,0.00,"[RENT, OWN, MORTGAGE, OTHER, NONE]"
8,annual_inc,float64,0.001,0.0,0.0,31901,6.84,"[24000.0, 30000.0, 12252.0, 49200.0, 80000.0]"
9,verification_status,int64,0.000,0.0,31.791,3,0.00,"[2, 1, 0]"


### Redundant Features

Selanjutnya, terdapat masalah redundansi antara `grade` dan `sub_grade` dalam arti informasinya; `sub_grade` sudah mencakup `grade` dengan informasi level yang lebih detil. Saya akan memilih `sub_grade` karena level detil informasinya lebih tinggi dibanding `grade`, dengan harapan dapat meningkatkan performa dari model.

In [68]:
print(f'Grade values: {df['grade'].unique()}')
print(f'\nSubgrade values: {df['sub_grade'].unique()}')

Grade values: ['B' 'C' 'A' 'E' 'F' 'D' 'G']

Subgrade values: ['B2' 'C4' 'C5' 'C1' 'B5' 'A4' 'E1' 'F2' 'C3' 'B1' 'D1' 'A1' 'B3' 'B4'
 'C2' 'D2' 'A3' 'A5' 'D5' 'A2' 'E4' 'D3' 'D4' 'F3' 'E3' 'F4' 'F1' 'E5'
 'G4' 'E2' 'G3' 'G2' 'G1' 'F5' 'G5']


In [69]:
sub_grade_order = [
    'A1', 'A2', 'A3', 'A4', 'A5',
    'B1', 'B2', 'B3', 'B4', 'B5',
    'C1', 'C2', 'C3', 'C4', 'C5',
    'D1', 'D2', 'D3', 'D4', 'D5',
    'E1', 'E2', 'E3', 'E4', 'E5',
    'F1', 'F2', 'F3', 'F4', 'F5',
    'G1', 'G2', 'G3', 'G4', 'G5'
]
df['sub_grade'] = pd.Categorical(
    df['sub_grade'], categories=sub_grade_order, ordered=True
).codes
skim_result = utils.skim_data(df)

skim_result

Total duplicate rows: 0
DF shape: (466285, 29)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,loan_amnt,int64,0.000,0.0,0.0,1352,0.29,"[5000, 2500, 2400, 10000, 3000]"
1,funded_amnt,int64,0.000,0.0,0.0,1354,0.29,"[5000, 2500, 2400, 10000, 3000]"
2,funded_amnt_inv,float64,0.000,0.0,0.05,9854,2.11,"[4975.0, 2500.0, 2400.0, 10000.0, 3000.0]"
3,int_rate,float64,0.000,0.0,0.0,506,0.11,"[10.65, 15.27, 15.96, 13.49, 12.69]"
4,installment,float64,0.000,0.0,0.0,55622,11.93,"[162.87, 59.83, 84.33, 339.31, 67.79]"
5,grade,object,0.000,-,-,7,0.00,"[B, C, A, E, F]"
6,sub_grade,int8,0.000,0.0,2.261,35,0.01,"[6, 13, 14, 10, 9]"
7,home_ownership,object,0.000,-,-,6,0.00,"[RENT, OWN, MORTGAGE, OTHER, NONE]"
8,annual_inc,float64,0.001,0.0,0.0,31901,6.84,"[24000.0, 30000.0, 12252.0, 49200.0, 80000.0]"
9,verification_status,int64,0.000,0.0,31.791,3,0.00,"[2, 1, 0]"


In [70]:
df = df.drop(columns=['grade'])
skim_result = utils.skim_data(df)

skim_result

Total duplicate rows: 0
DF shape: (466285, 28)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,loan_amnt,int64,0.000,0.0,0.0,1352,0.29,"[5000, 2500, 2400, 10000, 3000]"
1,funded_amnt,int64,0.000,0.0,0.0,1354,0.29,"[5000, 2500, 2400, 10000, 3000]"
2,funded_amnt_inv,float64,0.000,0.0,0.05,9854,2.11,"[4975.0, 2500.0, 2400.0, 10000.0, 3000.0]"
3,int_rate,float64,0.000,0.0,0.0,506,0.11,"[10.65, 15.27, 15.96, 13.49, 12.69]"
4,installment,float64,0.000,0.0,0.0,55622,11.93,"[162.87, 59.83, 84.33, 339.31, 67.79]"
5,sub_grade,int8,0.000,0.0,2.261,35,0.01,"[6, 13, 14, 10, 9]"
6,home_ownership,object,0.000,-,-,6,0.00,"[RENT, OWN, MORTGAGE, OTHER, NONE]"
7,annual_inc,float64,0.001,0.0,0.0,31901,6.84,"[24000.0, 30000.0, 12252.0, 49200.0, 80000.0]"
8,verification_status,int64,0.000,0.0,31.791,3,0.00,"[2, 1, 0]"
9,loan_status,object,0.000,-,-,9,0.00,"[Fully Paid, Charged Off, Current, Default, La..."


### Defining "Good Loan" vs. "Bad Loan"

Tujuan utama model yang akan dibuat adalah untuk menjawal pertanyaan bisnis sederhana: "apakah nasabah ini akan Gagal Bayar di masa depan", yang membuat permasalahan ini menjadi masalah klasifikasi biner.

Dalam `loan_status` sekarang, ada sembilan nilai unik yang harus dipilah agar dapat menjadi 1 (Gagal Bayar) atau 0 (Sukses Bayar):

In [71]:
df['loan_status'].unique()

array(['Fully Paid', 'Charged Off', 'Current', 'Default',
       'Late (31-120 days)', 'In Grace Period', 'Late (16-30 days)',
       'Does not meet the credit policy. Status:Fully Paid',
       'Does not meet the credit policy. Status:Charged Off'],
      dtype=object)

Untuk memilih kelompok Gagal Bayar, saya akan memilih semua variasi `Charged Off`, `Default`, dan variasi `Late (x days)`, karena saya ingin model ini secara proaktif mengidentifikasi calon nasabah yang berpotensi menyebabkan kerugian finansial bagi perusahaan.

Untuk memilih kelompok Sukses Bayar, saya akan memilih semua variasi `Fully Paid`, karena ini adalah mayoritas nasabah yang baik dan menguntungkan. Model harus bisa membedakan mereka dari kelompok berisiko.

Baris data yang tidak tergolong dalam dua kelompok di atas akan dibuang.

In [72]:
print(f'DataFrame sizes before selection: {df.shape}\n')
good_loan_statuses = [
    'Fully Paid', 
    'Does not meet the credit policy. Status:Fully Paid'
]
bad_loan_statuses = [
    'Charged Off', 
    'Default', 
    'Late (31-120 days)', 
    'Late (16-30 days)',
    'Does not meet the credit policy. Status:Charged Off'
]
df = df[df['loan_status'].isin(good_loan_statuses + bad_loan_statuses)].copy()
df['loan_status_binary'] = df['loan_status'].apply(
    lambda x: 1 if x in bad_loan_statuses else 0
)
# remove loan_status and replace with loan_status_binary
df = df.drop(columns=['loan_status'])
df = df.rename(columns={'loan_status_binary': 'loan_status'})
skim_result = utils.skim_data(df)

skim_result

DataFrame sizes before selection: (466285, 28)

Total duplicate rows: 0
DF shape: (238913, 28)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,loan_amnt,int64,0.000,0.0,0.0,1310,0.55,"[5000, 2500, 2400, 10000, 3000]"
1,funded_amnt,int64,0.000,0.0,0.0,1313,0.55,"[5000, 2500, 2400, 10000, 3000]"
2,funded_amnt_inv,float64,0.000,0.0,0.098,9560,4.00,"[4975.0, 2500.0, 2400.0, 10000.0, 5000.0]"
3,int_rate,float64,0.000,0.0,0.0,505,0.21,"[10.65, 15.27, 15.96, 13.49, 7.9]"
4,installment,float64,0.000,0.0,0.0,43848,18.35,"[162.87, 59.83, 84.33, 339.31, 156.46]"
5,sub_grade,int8,0.000,0.0,2.303,35,0.01,"[6, 13, 14, 10, 3]"
6,home_ownership,object,0.000,-,-,6,0.00,"[RENT, OWN, MORTGAGE, OTHER, NONE]"
7,annual_inc,float64,0.002,0.0,0.0,18715,7.83,"[24000.0, 30000.0, 12252.0, 49200.0, 36000.0]"
8,verification_status,int64,0.000,0.0,34.749,3,0.00,"[2, 1, 0]"
9,purpose,object,0.000,-,-,14,0.01,"[credit_card, car, small_business, other, wedd..."


Terlihat ada pengurangan sebanyak 226.721 baris data dari `loan_status` berlabel `Current` dan `In Grace Period`. Sekilas ini terlihat seperti pemborosan data, tapi saya berpegang teguh bahwa kualitas data lebih penting daripada kuantitas data. Data tersebut adalah noise, dan mempertahankan noise akan berakibat fatal yang dapat merusak kemampuan model untuk belajar.

## Splitting Data

In [73]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['loan_status'])
y = df[['loan_status']]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=29, stratify=y
)

In [74]:
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')

X_train shape: (191130, 27)
X_test shape: (47783, 27)


In [75]:
df_train = pd.concat([X_train, y_train], axis=1)
df_train.to_parquet('input/df_train.parquet', index=False)
df_test = pd.concat([X_test, y_test], axis=1)
df_test.to_parquet('input/df_test.parquet', index=False)